# SuperAI Season 6 - Individual Hackathon - Chest Disease Detection

Pipeline: **MedCLIP (ViT) fine-tune** -> multi-label 13 findings -> threshold -> binary 0/1

> **Metric ของ competition = sample-average F1** (F1 ต่อรูป แล้วเฉลี่ยทุกรูป = `f1_score(average="samples")`).
> **Baseline = 0.57009**

**Flow:** Setup Data -> Data Exploration (EDA) -> Preprocessing & Transforms -> Training (MedCLIP) -> Evaluation -> Submission

> รันบน **Kaggle** (เปิด GPU + Internet) หรือ Colab ก็ได้ — path/device auto-detect.
> Backbone = MedCLIP (pretrain บน CheXpert+MIMIC ตรงโดเมนนี้); ถ้าโหลด weight ไม่ได้ จะ fallback BiomedCLIP -> DenseNet121 อัตโนมัติ

## Setup Data

In [ ]:
# transformers จะลาก TensorFlow มา deadlock ตอน import medclip บนบางเครื่อง -> ปิดไว้ก่อน import ใด ๆ
import os
os.environ.setdefault("USE_TF", "0")
os.environ.setdefault("USE_FLAX", "0")
os.environ.setdefault("TRANSFORMERS_NO_ADVISORY_WARNINGS", "1")
!pip install -q medclip open_clip_torch timm scikit-learn matplotlib seaborn tqdm

In [ ]:
# ============================================================
# Setup Data — รองรับ 3 กรณี: (1) Kaggle input ที่ mount ไว้  (2) zip ในเครื่อง  (3) ดาวน์โหลดเอง
# ============================================================
import os, glob, subprocess

COMP_NAME = "individual-test-chest-disease-detection"

def _find_root():
    cands = ["/kaggle/input/" + COMP_NAME] + glob.glob("/kaggle/input/*")
    cands += ["dataset", "comp_data", "."]
    for c in cands:
        if c and os.path.exists(os.path.join(c, "train.csv")):
            return c
    return None

DATA_ROOT = _find_root()
if DATA_ROOT is None:
    # ไม่เจอ -> ลอง unzip ไฟล์ zip ในโฟลเดอร์ ถ้าไม่มีค่อยดาวน์โหลดจาก Kaggle
    zips = sorted(glob.glob("*.zip"), key=os.path.getsize, reverse=True)
    if not zips:
        if not os.path.exists(os.path.expanduser("~/.kaggle/kaggle.json")):
            try:
                from google.colab import files
                print("Upload your kaggle.json:"); files.upload()
                os.makedirs(os.path.expanduser("~/.kaggle"), exist_ok=True)
                os.replace("kaggle.json", os.path.expanduser("~/.kaggle/kaggle.json"))
                os.chmod(os.path.expanduser("~/.kaggle/kaggle.json"), 0o600)
            except Exception as e:
                print("need kaggle.json:", e)
        subprocess.run(["kaggle", "competitions", "download", "-c", COMP_NAME], check=False)
        zips = sorted(glob.glob("*.zip"), key=os.path.getsize, reverse=True)
    if zips:
        os.makedirs("dataset", exist_ok=True)
        subprocess.run(["unzip", "-qo", zips[0], "-d", "dataset"], check=False)
    DATA_ROOT = _find_root()

assert DATA_ROOT is not None, "train.csv not found — set DATA_ROOT manually."
print("DATA_ROOT:", DATA_ROOT)

### Imports & helpers

In [ ]:
import random, time, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image
import torch, torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import train_test_split
from tqdm.auto import tqdm
warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 20)

SEED = 42
def seed_everything(s):
    random.seed(s); np.random.seed(s); torch.manual_seed(s)
    if torch.cuda.is_available(): torch.cuda.manual_seed_all(s)
seed_everything(SEED)

# DataLoader workers: macOS 'spawn' re-imports a flat script; 'fork' is fine for CPU workers
import multiprocessing as _mp
try:
    if _mp.get_start_method(allow_none=True) != "fork": _mp.set_start_method("fork", force=True)
except (RuntimeError, ValueError):
    pass

if torch.cuda.is_available():            DEVICE = torch.device("cuda")
elif getattr(torch.backends,"mps",None) and torch.backends.mps.is_available():
                                         DEVICE = torch.device("mps")
else:                                    DEVICE = torch.device("cpu")
USE_AMP = DEVICE.type == "cuda"
print("Device:", DEVICE)

from sklearn.metrics import f1_score

def mean_column_auc(y_true, y_prob):
    # secondary metric: mean of per-class ROC-AUC (good for tracking ranking quality)
    aucs = [roc_auc_score(y_true[:, c], y_prob[:, c]) for c in range(y_true.shape[1])
            if len(np.unique(y_true[:, c])) > 1]
    return (float(np.mean(aucs)) if aucs else float("nan")), aucs

# ---- competition metric = sample-average F1 (needs BINARY predictions) ----
def samples_f1(y_true, y_bin):
    return f1_score(y_true, y_bin, average="samples", zero_division=0)

def to_binary(prob, thr):
    # threshold to 0/1; guarantee >=1 label per image (every train image has >=1) via argmax fallback
    b = (prob >= thr).astype(int)
    empty = b.sum(1) == 0
    if empty.any():
        b[empty, prob[empty].argmax(1)] = 1
    return b

def best_threshold(y_true, prob, lo=0.05, hi=0.60, step=0.01):
    ths = np.arange(lo, hi + 1e-9, step)
    f1s = [samples_f1(y_true, to_binary(prob, t)) for t in ths]
    i = int(np.argmax(f1s))
    return float(ths[i]), float(f1s[i]), ths, np.array(f1s)

### Load Data

In [ ]:
def find_images_dir(root):
    for pat in ["images/images", "images", "."]:
        d = os.path.join(root, pat)
        if glob.glob(os.path.join(d, "*.jpg")):
            return d
    for d, _, files in os.walk(root):
        if any(f.lower().endswith(".jpg") for f in files):
            return d
    raise FileNotFoundError("no .jpg under " + root)

IMAGES_DIR = find_images_dir(DATA_ROOT)
train = pd.read_csv(os.path.join(DATA_ROOT, "train.csv"))
LABELS = [c for c in train.columns if c != "filename"]
N_CLASSES = len(LABELS)
train[LABELS] = train[LABELS].fillna(0).clip(0, 1)

disk = {os.path.basename(p): p for p in glob.glob(os.path.join(IMAGES_DIR, "*.jpg"))}
train = train[train["filename"].isin(disk)].reset_index(drop=True)

# Official submission template: some rows are PRE-FILLED (keep as-is), the rest are
# EMPTY and must be predicted (exactly like heart_disease.ipynb fills only NaN rows).
SAMPLE_SUB = pd.read_csv(os.path.join(DATA_ROOT, "test_submission.csv"))
PREDICT_MASK = SAMPLE_SUB[LABELS].isna().all(axis=1)        # rows to predict
test_files = SAMPLE_SUB.loc[PREDICT_MASK, "filename"].tolist()

print(f"Images dir : {IMAGES_DIR}")
print(f"Classes ({N_CLASSES}): {LABELS}")
print(f"Train (labeled): {len(train)}")
print(f"Submission template: {len(SAMPLE_SUB)} rows | to predict: {len(test_files)} | given: {int((~PREDICT_MASK).sum())}")
train.head()

## Data Exploration (EDA)

### Label distribution (positive rate per class) — imbalanced

In [ ]:
pos_rate = train[LABELS].mean().sort_values()
fig, ax = plt.subplots(figsize=(8, 5))
pos_rate.plot(kind="barh", ax=ax, color="salmon")
ax.set_title("Positive rate per finding (multi-label, imbalanced)")
ax.set_xlabel("fraction of images positive")
for i, v in enumerate(pos_rate.values): ax.text(v + 0.003, i, f"{v:.1%}", va="center", fontsize=8)
plt.tight_layout(); plt.show()

### Number of findings per image

In [ ]:
counts = train[LABELS].sum(axis=1)
fig, ax = plt.subplots(figsize=(7, 3))
counts.value_counts().sort_index().plot(kind="bar", ax=ax, color="steelblue")
ax.set_title("How many findings does one X-ray have?")
ax.set_xlabel("# positive labels per image"); ax.set_ylabel("count")
plt.tight_layout(); plt.show()
print("avg findings/image:", round(counts.mean(), 2), "| images with 0 findings:", int((counts==0).sum()))

### Label co-occurrence (how often findings appear together)

In [ ]:
M = train[LABELS].T.dot(train[LABELS]).astype(int)   # 13x13 co-occurrence
fig, ax = plt.subplots(figsize=(9, 7))
sns.heatmap(M, annot=True, fmt="d", cmap="YlOrRd", cbar=False,
            xticklabels=LABELS, yticklabels=LABELS, ax=ax, annot_kws={"size": 7})
ax.set_title("Label co-occurrence counts"); plt.xticks(rotation=45, ha="right")
plt.tight_layout(); plt.show()

### Sample chest X-rays

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(16, 8)); axes = axes.flatten()
for ax, (_, row) in zip(axes, train.sample(8, random_state=SEED).iterrows()):
    img = Image.open(disk[row["filename"]]).convert("L")
    findings = [l for l in LABELS if row[l] == 1] or ["(none)"]
    ax.imshow(img, cmap="gray"); ax.axis("off")
    ax.set_title("\n".join(findings), fontsize=8)
plt.suptitle("Sample chest X-rays with labels", y=1.02); plt.tight_layout(); plt.show()

## Preprocessing & Transforms

In [ ]:
import torchvision.transforms as T
IMG_SIZE = 224
MED_MEAN = [0.5862785803043838] * 3   # MedCLIP grayscale-CXR normalization
MED_STD  = [0.27950088968644304] * 3

def build_transforms(mean, std):
    # NO horizontal flip — laterality matters in CXR (heart/aorta side)
    train_tf = T.Compose([
        T.Resize((IMG_SIZE, IMG_SIZE)),
        T.RandomAffine(degrees=7, translate=(0.04, 0.04), scale=(0.96, 1.04)),
        T.ColorJitter(brightness=0.12, contrast=0.12),
        T.ToTensor(), T.Normalize(mean, std)])
    eval_tf = T.Compose([T.Resize((IMG_SIZE, IMG_SIZE)), T.ToTensor(), T.Normalize(mean, std)])
    return train_tf, eval_tf

class CXRDataset(Dataset):
    def __init__(self, files, labels, tf):
        self.files, self.labels, self.tf = list(files), labels, tf
    def __len__(self): return len(self.files)
    def __getitem__(self, i):
        x = self.tf(Image.open(disk[self.files[i]]).convert("RGB"))
        if self.labels is None: return x, self.files[i]
        return x, torch.tensor(self.labels[i], dtype=torch.float32)

## Model — MedCLIP backbone + classification head

In [ ]:
BACKBONE = os.environ.get("BACKBONE", "auto")   # auto | medclip-vit | biomedclip | densenet121

def build_backbone(name_pref):
    order = [name_pref] if name_pref != "auto" else ["medclip-vit", "biomedclip", "densenet121"]
    for name in order:
        try:
            if name.startswith("medclip"):
                from medclip import MedCLIPModel, MedCLIPVisionModelViT
                # MedCLIP checkpoint: pickled on CUDA + has a stale BERT 'position_ids' key.
                _ld, _lsd = torch.load, torch.nn.Module.load_state_dict
                _map = "cuda" if torch.cuda.is_available() else "cpu"
                torch.load = lambda *a, **k: (_ld(*a, **{**k, "map_location": k.get("map_location", _map)}))
                torch.nn.Module.load_state_dict = lambda self, sd, strict=True, *a, **k: _lsd(self, sd, False, *a, **k)
                try:
                    full = MedCLIPModel(vision_cls=MedCLIPVisionModelViT); full.from_pretrained()
                finally:
                    torch.load, torch.nn.Module.load_state_dict = _ld, _lsd
                class Enc(nn.Module):
                    def __init__(s, m): super().__init__(); s.m = m
                    def forward(s, x):
                        o = s.m(pixel_values=x); return o[0] if isinstance(o, (tuple, list)) else o
                enc = Enc(full.vision_model)
                with torch.no_grad(): fd = enc(torch.zeros(1, 3, IMG_SIZE, IMG_SIZE)).shape[1]
                print(f"[backbone] MedCLIP-ViT, feat_dim={fd}")
                return enc, fd, MED_MEAN, MED_STD, name
            elif name == "biomedclip":
                import open_clip
                m, _, _ = open_clip.create_model_and_transforms(
                    "hf-hub:microsoft/BiomedCLIP-PubMedBERT_256-vit_base_patch16_224")
                class Enc(nn.Module):
                    def __init__(s, v): super().__init__(); s.v = v
                    def forward(s, x): return s.v(x)
                enc = Enc(m.visual)
                with torch.no_grad(): fd = enc(torch.zeros(1, 3, IMG_SIZE, IMG_SIZE)).shape[1]
                print(f"[backbone] BiomedCLIP, feat_dim={fd}")
                return enc, fd, [0.48145466,0.4578275,0.40821073], [0.26862954,0.26130258,0.27577711], name
            elif name == "densenet121":
                import timm
                m = timm.create_model("densenet121", pretrained=True, num_classes=0, global_pool="avg")
                print(f"[backbone] DenseNet121, feat_dim={m.num_features}")
                return m, m.num_features, [0.485,0.456,0.406], [0.229,0.224,0.225], name
        except Exception as e:
            print(f"[backbone] {name} unavailable ({type(e).__name__}: {e}); next...")
    raise RuntimeError("no backbone available")

class CXRClassifier(nn.Module):
    def __init__(self, enc, feat_dim, n_classes):
        super().__init__(); self.encoder = enc
        self.head = nn.Sequential(nn.Dropout(0.2), nn.Linear(feat_dim, n_classes))
    def forward(self, x):
        f = self.encoder(x)
        if f.ndim > 2: f = f.flatten(1)
        return self.head(f)

encoder, FEAT_DIM, MEAN, STD, BACKBONE_USED = build_backbone(BACKBONE)
model = CXRClassifier(encoder, FEAT_DIM, N_CLASSES).to(DEVICE)
train_tf, eval_tf = build_transforms(MEAN, STD)
print("Backbone in use:", BACKBONE_USED)

## Training — fine-tune, select best by val sample-F1

In [ ]:
EPOCHS  = int(os.environ.get("EPOCHS", "6"))
BATCH   = int(os.environ.get("BATCH", "32"))
# macOS + MPS: forking DataLoader workers after the MPS context inits crashes them,
# so default to 0 workers there. Linux/Kaggle (CUDA/CPU) can use parallel workers.
WORKERS = int(os.environ.get("WORKERS", "0" if DEVICE.type == "mps" else "2"))

y_all = train[LABELS].values.astype("float32")
strat = (y_all.sum(1) > 0).astype(int)
tr_idx, va_idx = train_test_split(np.arange(len(train)), test_size=0.10,
                                  stratify=strat, random_state=SEED)
pin = DEVICE.type == "cuda"
train_loader = DataLoader(CXRDataset(train["filename"].values[tr_idx], y_all[tr_idx], train_tf),
                          batch_size=BATCH, shuffle=True, num_workers=WORKERS, pin_memory=pin)
val_loader   = DataLoader(CXRDataset(train["filename"].values[va_idx], y_all[va_idx], eval_tf),
                          batch_size=BATCH, shuffle=False, num_workers=WORKERS, pin_memory=pin)

# class imbalance -> positive weighting (capped)
pos = y_all[tr_idx].sum(0); neg = len(tr_idx) - pos
pos_weight = torch.tensor(np.clip(neg/np.clip(pos,1,None), 1.0, 10.0), dtype=torch.float32).to(DEVICE)
criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
optimizer = torch.optim.AdamW([
    {"params": model.encoder.parameters(), "lr": 1e-5},
    {"params": model.head.parameters(),    "lr": 1e-4}], weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=max(1, EPOCHS))
scaler = torch.cuda.amp.GradScaler(enabled=USE_AMP)

@torch.no_grad()
def evaluate(loader):
    model.eval(); P, Y = [], []
    for x, y in tqdm(loader, desc="validate", leave=False):
        x = x.to(DEVICE)
        P.append(torch.sigmoid(model(x)).float().cpu().numpy()); Y.append(y.numpy())
    P, Y = np.concatenate(P), np.concatenate(Y)
    return mean_column_auc(Y, P)[0], P, Y

history, best_f1, best_state = [], -1.0, None
for epoch in range(1, EPOCHS + 1):
    model.train(); t0 = time.time(); run = 0.0; seen = 0
    pbar = tqdm(train_loader, desc=f"epoch {epoch}/{EPOCHS}")
    for x, y in pbar:
        x, y = x.to(DEVICE), y.to(DEVICE)
        optimizer.zero_grad()
        with torch.autocast(device_type="cuda", enabled=USE_AMP):
            loss = criterion(model(x), y)
        scaler.scale(loss).backward(); scaler.step(optimizer); scaler.update()
        run += loss.item() * x.size(0); seen += x.size(0)
        pbar.set_postfix(loss=f"{run/seen:.4f}")   # live running loss
    scheduler.step()
    val_auc, vp, vy = evaluate(val_loader)
    thr, vf1, _, _ = best_threshold(vy, vp)          # tune threshold for sample-F1
    history.append((val_auc, vf1))
    print(f"epoch {epoch}/{EPOCHS} | loss {run/len(tr_idx):.4f} | "
          f"val mAUC {val_auc:.4f} | val sampleF1 {vf1:.4f} (thr={thr:.2f}) | {time.time()-t0:.0f}s")
    if vf1 > best_f1:                                 # select best by the REAL metric (F1)
        best_f1 = vf1
        best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}

if best_state is not None: model.load_state_dict(best_state)
print(f"\nBest validation sample-average F1 = {best_f1:.4f}")

## Evaluation

### Training curve (val sample-F1 per epoch)

In [ ]:
hist = np.array(history)   # columns: (mean AUC, sample-F1)
fig, ax = plt.subplots(figsize=(7, 3))
ax.plot(range(1, len(hist)+1), hist[:, 1], marker="o", label="val sample-F1", color="seagreen")
ax.plot(range(1, len(hist)+1), hist[:, 0], marker="s", label="val mean AUC", color="steelblue", alpha=0.6)
ax.set_xlabel("epoch"); ax.set_ylabel("score"); ax.set_title("Validation curves"); ax.legend()
plt.tight_layout(); plt.show()

### Threshold tuning — sample-average F1 (the competition metric)

In [ ]:
# get validation probabilities from the trained model, then sweep the threshold
val_auc, val_prob, val_true = evaluate(val_loader)
BEST_THR, BEST_F1, ths, f1s = best_threshold(val_true, val_prob)

fig, ax = plt.subplots(figsize=(8, 3))
ax.plot(ths, f1s)
ax.axvline(BEST_THR, color="red", ls="--", label=f"best thr = {BEST_THR:.2f}")
ax.axhline(0.57009, color="grey", ls=":", label="baseline 0.57009")
ax.set_xlabel("threshold"); ax.set_ylabel("sample-average F1")
ax.set_title("F1 threshold tuning"); ax.legend(); plt.tight_layout(); plt.show()

print(f"Validation sample-average F1 = {BEST_F1:.4f}  @ threshold {BEST_THR:.2f}")
print(f"(for reference: validation mean AUC = {val_auc:.4f}; baseline to beat = 0.57009)")

### Per-class F1 on validation (at the tuned threshold)

In [ ]:
val_bin = to_binary(val_prob, BEST_THR)
per_f1 = {lab: f1_score(val_true[:, c], val_bin[:, c], zero_division=0) for c, lab in enumerate(LABELS)}
s = pd.Series(per_f1).sort_values()
fig, ax = plt.subplots(figsize=(8, 5))
s.plot(kind="barh", ax=ax, color="steelblue")
ax.set_title(f"Per-class F1 @ thr={BEST_THR:.2f} (sample-avg F1 = {BEST_F1:.4f})")
for i, v in enumerate(s.values): ax.text(v+0.005, i, f"{v:.3f}", va="center", fontsize=8)
plt.tight_layout(); plt.show()

## Generate Submission

In [ ]:
test_loader = DataLoader(CXRDataset(test_files, None, eval_tf),
                         batch_size=BATCH, shuffle=False, num_workers=WORKERS, pin_memory=pin)
@torch.no_grad()
def predict(loader):
    model.eval(); probs, names = [], []
    for x, fns in tqdm(loader, desc="predict test"):
        probs.append(torch.sigmoid(model(x.to(DEVICE))).float().cpu().numpy()); names += list(fns)
    return np.concatenate(probs), names

test_probs, test_names = predict(test_loader)

# metric = sample-average F1 -> BINARY 0/1 at the tuned threshold
pred_bin = to_binary(test_probs, BEST_THR)
bmap = {fn: row for fn, row in zip(test_names, pred_bin)}

# Fill ONLY the empty rows of the official template; keep pre-filled rows as-is.
submission = SAMPLE_SUB.copy()
for i in submission.index[PREDICT_MASK]:
    submission.loc[i, LABELS] = bmap[submission.at[i, "filename"]]

# sanity: template structure, no NaN, only 0/1
assert list(submission.columns) == ["filename"] + LABELS
assert submission[LABELS].isna().sum().sum() == 0
assert submission[LABELS].isin([0, 1, 0.0, 1.0]).all().all()

submission.to_csv("submission.csv", index=False)
print(f"Saved submission.csv  shape={submission.shape} | backbone={BACKBONE_USED}")
print(f"val sample-F1={BEST_F1:.4f} @ thr={BEST_THR:.2f} | "
      f"filled {int(PREDICT_MASK.sum())} rows, kept {int((~PREDICT_MASK).sum())} given")
submission.head(10)

### Submit
```bash
kaggle competitions submit -c individual-test-chest-disease-detection -f submission.csv -m "MedCLIP-ViT fine-tune"
```
**ดันคะแนนต่อ:** เพิ่ม `EPOCHS`, ทำ TTA (เฉลี่ยหลาย crop), หรือ ensemble MedCLIP + DenseNet121.